# Train a Tokenizer in Forty Lines

**Topic:** Byte-pair encoding &nbsp;·&nbsp; **Difficulty:** Medium

BPE is the least glamorous algorithm in modern NLP and one of the most
consequential: it decides what a "token" is, and therefore what the model can
count, spell, and do arithmetic on. The training loop is embarrassingly simple:
count adjacent pairs, merge the most frequent, repeat.

Implement it, then use the learned merges to encode held-out text and watch a
word the tokenizer has never seen get shattered into pieces.

## The data (created in the next cell)

- `corpus`: a list of words with frequencies, pre-split into characters.
- `n_merges = 60`.
- `test_words`, including one word made of familiar characters in an unfamiliar
  order.

## Your task

With **plain Python and NumPy**:

1. Represent each word as a tuple of symbols, starting from characters plus an
   end-of-word marker `"</w>"`.
2. Repeat `n_merges` times: count every adjacent symbol pair weighted by word
   frequency, take the most frequent (break ties by the pair's sorted order to
   stay deterministic), and merge it everywhere.
3. Encode each test word greedily by applying the learned merges in the order
   they were learned.
4. Set `answers` to a dict with `"n_merges_done"` (int), `"vocab_size"` (int,
   distinct symbols after training), `"tokens_per_test_word"` (an array of token
   counts for `test_words`), and `"total_test_tokens"` (int).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
also checks the rare word costs more tokens than the common ones.

In [ ]:
import numpy as np

words = ["low", "lower", "lowest", "new", "newer", "newest", "wide", "wider",
         "widest", "slow", "slower", "slowest", "show", "shower", "known",
         "grow", "grower", "flow", "flower", "glow"]
freqs = [50, 30, 20, 45, 25, 15, 40, 22, 12, 35, 18, 10, 28, 14, 20,
         33, 16, 30, 15, 24]
corpus = list(zip(words, freqs))
n_merges = 60
test_words = ["lowest", "slower", "newest", "wlgnoshrew"]
print(f"{len(corpus)} training words, {n_merges} merges, "
      f"{len(test_words)} test words")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("n_merges_done", [], 39.0, -6.530082577765468, 1e-06),
    ("vocab_size", [], 20.0, -3.348760296289983, 1e-06),
    ("tokens_per_test_word", [4], 12.0, 12.564765784749664, 1e-06),
    ("total_test_tokens", [], 12.0, -2.00925617777399, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "tokens_per_test_word" in answers:
    _c = np.asarray(answers["tokens_per_test_word"], dtype=float)
    _rare = _c[-1] > 2 * _c[:-1].max()
    print(f"  {'rare word costs more':<24} {'ok' if _rare else 'wrong'}  "
          f"(counts {_c.astype(int).tolist()})")
    _ok = _ok and _rare

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The merges learned here are the frequent suffixes (`est</w>`, `er</w>`, `low`)
so common words collapse to two or three tokens. The scrambled word uses only
familiar characters but none of the familiar sequences, so it falls back to
near-character-level and costs several times more tokens. That is exactly what
happens to rare names, code identifiers, and non-English text in a
English-trained tokenizer, and it is why token counts (and therefore cost) vary
so much across languages.

```python
import numpy as np
from collections import Counter

vocab = {tuple(list(w) + ["</w>"]): f for w, f in corpus}

merges = []
for _ in range(n_merges):
    pairs = Counter()
    for sym, f in vocab.items():
        for i in range(len(sym) - 1):
            pairs[(sym[i], sym[i + 1])] += f
    if not pairs:
        break
    best_count = max(pairs.values())
    best = sorted([p for p, c in pairs.items() if c == best_count])[0]
    merges.append(best)
    new_vocab = {}
    for sym, f in vocab.items():
        out, i = [], 0
        while i < len(sym):
            if i < len(sym) - 1 and (sym[i], sym[i + 1]) == best:
                out.append(sym[i] + sym[i + 1])
                i += 2
            else:
                out.append(sym[i])
                i += 1
        new_vocab[tuple(out)] = new_vocab.get(tuple(out), 0) + f
    vocab = new_vocab

def encode(word):
    sym = list(word) + ["</w>"]
    for a, b in merges:
        out, i = [], 0
        while i < len(sym):
            if i < len(sym) - 1 and sym[i] == a and sym[i + 1] == b:
                out.append(a + b)
                i += 2
            else:
                out.append(sym[i])
                i += 1
        sym = out
    return sym

counts = np.array([len(encode(w)) for w in test_words], dtype=float)
symbols = set()
for sym in vocab:
    symbols.update(sym)

answers = {
    "n_merges_done": int(len(merges)),
    "vocab_size": int(len(symbols)),
    "tokens_per_test_word": counts,
    "total_test_tokens": float(counts.sum()),
}
```